In [1]:
def initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose):
    num_evals_init = num_evals
    init_range = hi - lo
    if is_pos:
        mse, z, ghat = eval_fn(hi)
        while mse < epsilon_squared and num_evals > 0:
            hi += init_range
            mse, z, ghat = eval_fn(hi)
            num_evals -= 1
        bound_dict = {'lo':mse_z_ghat_0, 'hi':(mse, z, ghat)}
    else:
        mse, z, ghat = eval_fn(lo)
        while mse > epsilon_squared and num_evals > 0: # remember epsilon_squared will be negative in this case
            lo -= init_range
            mse, z, ghat = eval_fn(lo)
            num_evals -= 1
        bound_dict = {'lo':(mse, z, ghat), 'hi':mse_z_ghat_0}
    if num_evals == 0:
        raise ValueError('Exceeded number of allowable evaluations during initialization of search bounds.')
    elif verbose:
        print(f'Initial bounds: ({lo}, {hi})')
        print(f'{num_evals}/{num_evals_init} evaluations remaining after initialization.')
    return lo, hi, bound_dict, num_evals

def bisection_search(lo, hi , eval_fn, num_evals, epsilon_squared, mse_z_ghat_0, verbose, tol=0):
    is_pos = hi > 0
    lo, hi, bound_dict, num_evals = initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose)
    while lo < hi - tol and num_evals > 0:
        mid = (lo + hi) / 2
        mse, z, ghat = eval_fn(mid)
        if verbose:
            print(f'alpha:{mid}, mse:{mse}, lo:{lo}, hi:{hi}')
        if mse < epsilon_squared:
            lo = mid
            bound_dict['lo'] = (mse, z, ghat)
        elif mse > epsilon_squared:
            hi = mid
            bound_dict['hi'] = (mse, z, ghat)
        else:
            return (mid, mse, z, ghat)
        num_evals -= 1
    return (hi,) + bound_dict['hi'] if is_pos else (lo,) + bound_dict['lo']

            

# def optimize_alpha(vanilla_dy_dx, zo_dy_dx, net, criterion, method, gt_data, 
#                    label_pred, num_attack_iterations, num_dummy, imidx_list,
#                    num_alpha_search_iterations, epsilon_squared):
#      inv_attack_closure = lambda ghat: inv_attack(ghat, net, criterion, method, gt_data, label_pred, 
#                                        num_attack_iterations, None, num_dummy, None, 
#                                        imidx_list, None, False)
def inv_attack_closure(alpha):
    return alpha * alpha, 'z', 'ghat'
def get_bisection_search_eval_fn(sign):
    if sign == 'pos':
        def bisection_search_eval_fn(alpha):
            return inv_attack_closure(alpha)
    elif sign == 'neg':
        def bisection_search_eval_fn(alpha):
            mse, z, ghat = inv_attack_closure(alpha)
            return -mse, z, ghat
    else:
        raise ValueError('sign must be either `pos` or `neg')
    return bisection_search_eval_fn

            
epsilon_squared = 0.1
num_alpha_search_iterations = 10
mse_0, z_0, ghat_0 = inv_attack_closure(0)
if mse_0 >= epsilon_squared:
    # if the vanilla gradient (alpha=0) is already larger than the error tol, then we are satisfying the constraint and can't reduce alpha any further. return 
    print( 0, ghat_0)
alpha_star_pos, mse_star_pos, zhat_pos, ghat_alpha_star_pos = bisection_search(0, 1, get_bisection_search_eval_fn('pos'), num_alpha_search_iterations, epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True)
alpha_star_neg, mse_star_neg, zhat_neg, ghat_alpha_star_neg = bisection_search(-1, 0, get_bisection_search_eval_fn('neg'), num_alpha_search_iterations, -epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True)


Initial bounds: (0, 1)
10/10 evaluations remaining after initialization.
alpha:0.5, mse:0.25, lo:0, hi:1
alpha:0.25, mse:0.0625, lo:0, hi:0.5
alpha:0.375, mse:0.140625, lo:0.25, hi:0.5
alpha:0.3125, mse:0.09765625, lo:0.25, hi:0.375
alpha:0.34375, mse:0.1181640625, lo:0.3125, hi:0.375
alpha:0.328125, mse:0.107666015625, lo:0.3125, hi:0.34375
alpha:0.3203125, mse:0.10260009765625, lo:0.3125, hi:0.328125
alpha:0.31640625, mse:0.1001129150390625, lo:0.3125, hi:0.3203125
alpha:0.314453125, mse:0.09888076782226562, lo:0.3125, hi:0.31640625
alpha:0.3154296875, mse:0.09949588775634766, lo:0.314453125, hi:0.31640625
Initial bounds: (-1, 0)
10/10 evaluations remaining after initialization.
alpha:-0.5, mse:-0.25, lo:-1, hi:0
alpha:-0.25, mse:-0.0625, lo:-0.5, hi:0
alpha:-0.375, mse:-0.140625, lo:-0.5, hi:-0.25
alpha:-0.3125, mse:-0.09765625, lo:-0.375, hi:-0.25
alpha:-0.34375, mse:-0.1181640625, lo:-0.375, hi:-0.3125
alpha:-0.328125, mse:-0.107666015625, lo:-0.34375, hi:-0.3125
alpha:-0.3203125,

In [35]:
alpha_star_pos

0.31640625

In [36]:
alpha_star_neg

-0.31640625

# Assume the Objective is Noisy and Decide alpha via stochastic sampling and shrinking confidence interval

In [ ]:
import scipy
stats = scipy.stats
import numpy as np
from matplotlib import pyplot as plt
import time
import json
from scipy.stats import norm, beta
import os

def cp_ci(v, n, delta):
    # two-sided Clopper–Pearson with edge cases
    if n == 0:
        raise ValueError("n must be >= 1")
    if v == 0:
        theta_l = 0.0
        theta_u = beta.ppf(1 - delta/2, 1, n)
    elif v == n:
        theta_l = beta.ppf(delta/2, n, 1)
        theta_u = 1.0
    else:
        theta_l = beta.ppf(delta/2, v, n - v + 1)
        theta_u = beta.ppf(1 - delta/2, v + 1, n - v)
    return float(theta_l), float(theta_u)

def plot_ci_step(out_path, alpha, tau, delta, eps2, noise_var, log_rows, step_idx):
    """
    log_rows: list of dicts with keys:
      n, v, theta_l, theta_u, mse (last), status (optional)
    """
    ns = [r["n"] for r in log_rows]
    L  = [r["theta_l"] for r in log_rows]
    U  = [r["theta_u"] for r in log_rows]
    v  = [r["v"] for r in log_rows]

    plt.figure()
    plt.plot(ns, L, marker="o", label=f"theta_L")
    plt.plot(ns, U, marker="o", label=f"theta_U")
    plt.axhline(tau, linestyle="--", label=f"tau ={tau} (target violation prob)")
    plt.ylim(-0.02, 1.02)
    plt.xlabel("n (samples at this alpha)")
    plt.ylabel(r"CI bounds for $\theta = P(\mathrm{MSE} < \varepsilon^2)$")
    plt.title(f"alpha={alpha:.6g}  delta={delta}  last v/n={v[-1]}/{ns[-1]}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_path, f"ci_step_{step_idx:03d}.png"))
    plt.close()

def get_ci_protection_output_logged(
    alpha, epsilon_squared, eval_fn, num_evals, num_init_samples,
    tau=0.1, delta=0.1, tol=1e-7,
    log_dir=None
):
    """
    Returns (status, num_evals_remaining).
    status in {"safe","unsafe","boundary","inconclusive"}.
    Also saves plots per sampling step if log_dir is provided.
    """
    if num_evals <= 0:
        return "inconclusive", num_evals, 0

    os.makedirs(log_dir, exist_ok=True) if log_dir else None

    # initial samples
    k0 = min(num_evals, num_init_samples)
    mses = [eval_fn(alpha)[0] for _ in range(k0)]
    num_evals -= k0

    log_rows = []
    v = int(np.sum(np.array(mses) < epsilon_squared))
    n = len(mses)
    theta_l, theta_u = cp_ci(v, n, delta)

    log_rows.append({"n": n, "v": v, "theta_l": theta_l, "theta_u": theta_u, "mse": float(mses[-1])})

    step_idx = k0
    # if log_dir:
    #     plot_ci_step(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
    #     with open(os.path.join(log_dir, "trace.jsonl"), "a") as f:
    #         f.write(json.dumps(log_rows[-1]) + "\n")

    # early decisive
    if tau > theta_u:
        plotit(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
        return "safe", num_evals, step_idx
    if tau < theta_l:
        plotit(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
        return "unsafe", num_evals, step_idx

    # sequential tightening
    while True:
        if (theta_u - theta_l) < tol:
            plotit(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
            return "boundary", num_evals, step_idx
        if num_evals <= 0:
            plotit(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
            return "inconclusive", num_evals, step_idx

        mses.append(eval_fn(alpha)[0])
        num_evals -= 1

        v = int(np.sum(np.array(mses) < epsilon_squared))
        n = len(mses)
        theta_l, theta_u = cp_ci(v, n, delta)

        step_idx += 1
        log_rows.append({"n": n, "v": v, "theta_l": theta_l, "theta_u": theta_u, "mse": float(mses[-1])})

        # if log_dir:
        #     plot_ci_step(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
        #     with open(os.path.join(log_dir, "trace.jsonl"), "a") as f:
        #         f.write(json.dumps(log_rows[-1]) + "\n")

        if tau > theta_u:
            plotit(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
            return "safe", num_evals, step_idx
        if tau < theta_l:
            plotit(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
            return "unsafe", num_evals, step_idx


def plotit(log_dir, alpha, tau, delta, epsilon_squared, noise_var, log_rows, step_idx):
    plot_ci_step(log_dir, alpha, tau, delta, epsilon_squared, None, log_rows, step_idx)
    with open(os.path.join(log_dir, "trace.jsonl"), "a") as f:
        f.write(json.dumps(log_rows[-1]) + "\n")

def plot_analytical_distribution(alpha, epsilon_squared, noise_var, delta, out_path):
    sigma = np.sqrt(noise_var)
    mu = alpha**2

    # key quantities
    theta = norm.cdf((epsilon_squared - mu)/sigma)          # P(Y < eps^2)
    q_delta = mu + sigma * norm.ppf(delta)                  # delta-quantile of Y

    # make a CDF plot (analytic)
    xs = np.linspace(mu - 5*sigma, mu + 5*sigma, 600)
    cdf = norm.cdf((xs - mu)/sigma)

    plt.figure()
    plt.plot(xs, cdf, label=f"Y ~ N({mu:.3g}, {sigma**2:.3g})")

    plt.axvline(epsilon_squared, linestyle="--", label=f"eps^2={epsilon_squared:.3g}")
    plt.axvline(q_delta, linestyle="--", label=f"q_delta (delta={delta})={q_delta:.3g}")

    plt.axhline(delta, linestyle=":", label=f"delta={delta}")
    plt.scatter([epsilon_squared], [theta], zorder=3)
    plt.text(epsilon_squared, theta, f"  theta=P(Y<eps^2)={theta:.3f}", va="center")

    plt.ylim(-0.02, 1.02)
    plt.xlabel("y")
    plt.ylabel("CDF")
    plt.title(f"alpha={alpha:.6g}  theta={theta:.3f}  q_delta={q_delta:.3g}")
    plt.legend()
    plt.tight_layout()
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(out_path)
    plt.close()

    # Decision sanity:
    # "unsafe" if theta > tau (too much mass below eps^2)
    return theta, q_delta



import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.stats import norm
from matplotlib.lines import Line2D

def plot_sequential_alpha_progression(
    alpha_history,
    status_history,
    eval_cnt_history,
    epsilon_squared,
    noise_var,
    tau,
    out_path=None
):
    """
    Plots bisection progression in visit order.

    x-axis: uniform iteration index
    tick labels: actual alpha values as strings
    y-axis: tau-quantile and epsilon_squared

    Points are color-coded by status:
        safe         -> green
        unsafe       -> red
        boundary     -> orange
        inconclusive -> gray

    Also annotates each point with the number of evals used at that alpha.
    """

    assert len(alpha_history) == len(status_history) == len(eval_cnt_history), \
        "alpha_history, status_history, eval_cnt_history must all be same length"

    sigma = np.sqrt(noise_var)
    z_tau = norm.ppf(tau)

    # Compute tau-quantile for each alpha
    q_tau_vals = [a**2 + sigma * z_tau for a in alpha_history]

    # Uniform x positions
    x_positions = np.arange(len(alpha_history))

    # Status color mapping
    color_map = {
        "safe": "green",
        "unsafe": "red",
        "boundary": "orange",
        "inconclusive": "gray"
    }
    colors = [color_map.get(s, "black") for s in status_history]

    plt.figure(figsize=(11, 5))

    # Plot quantile curve (line)
    plt.plot(x_positions, q_tau_vals, linestyle='-', alpha=0.4)

    # Plot epsilon line (no label here; legend is manual)
    plt.axhline(
        epsilon_squared,
        linestyle="--",
        linewidth=2,
        color="black"
    )

    # Plot colored decision markers + annotate eval counts above each
    for x, y, c, k in zip(x_positions, q_tau_vals, colors, eval_cnt_history):
        plt.scatter(x, y, color=c, s=80, zorder=3)
        plt.annotate(
            str(k),
            (x, y),
            textcoords="offset points",
            xytext=(0, 8),          # pixels above point
            ha="center",
            va="bottom",
            color="black",
            fontsize=9,             # tweak if you want bigger/smaller
            zorder=4
        )

    # Format tick labels as alpha strings
    alpha_labels = [f"{a:.5f}" for a in alpha_history]
    plt.xticks(x_positions, alpha_labels, rotation=45)

    plt.xlabel(r"$\alpha$")
    plt.ylabel("MSE")
    plt.title("Sequential Bisection Progression (τ-quantile vs ε²)")

    # Legend (manual)
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', label='safe',
               markerfacecolor='green', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='unsafe',
               markerfacecolor='red', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='boundary',
               markerfacecolor='orange', markersize=8),
        Line2D([0], [0], marker='o', color='w', label='eval budget exhausted',
               markerfacecolor='gray', markersize=8),

        # epsilon^2 line with numeric value
        Line2D([0], [0], linestyle='--', color='black',
               label=fr"$\varepsilon^2$ = {epsilon_squared:g}"),

        # text-only entry explaining the annotations
        Line2D([0], [0], linestyle='None', marker=None, color='black',
               label='num evals used')
    ]
    plt.legend(handles=legend_elements)

    plt.tight_layout()

    if out_path:
        os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
        plt.savefig(out_path)
        plt.close()
    else:
        plt.show()

In [ ]:


def initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose):
    num_evals_init = num_evals
    init_range = hi - lo
    if is_pos:
        mse, z, ghat = eval_fn(hi)
        num_evals -= 1
        while mse < epsilon_squared and num_evals > 0:
            hi += init_range
            mse, z, ghat = eval_fn(hi)
            num_evals -= 1
        bound_dict = {'lo':mse_z_ghat_0, 'hi':(mse, z, ghat)}
    else:
        mse, z, ghat = eval_fn(lo)
        num_evals -= 1
        while mse > epsilon_squared and num_evals > 0: # remember epsilon_squared will be negative in this case
            lo -= init_range
            mse, z, ghat = eval_fn(lo)
            num_evals -= 1
        bound_dict = {'lo':(mse, z, ghat), 'hi':mse_z_ghat_0}
    if num_evals <= 0:
        raise ValueError('Exceeded number of allowable evaluations during initialization of search bounds.')
    elif verbose:
        print(f'Initial bounds: ({lo}, {hi})')
        print(f'{num_evals}/{num_evals_init} evaluations remaining after initialization.')
    return lo, hi, bound_dict, num_evals


def get_v_statistic(mses, epsilon_squared):
    return int(sum(mses < epsilon_squared))

def get_ci_bounds(mses, epsilon_squared, delta):
    n = len(mses)
    if n == 0:
        raise ValueError("mses must be non-empty")
    v = get_v_statistic(np.array(mses), epsilon_squared)
    if v == 0:
        theta_l = 0
        theta_u = stats.beta.ppf(1 - delta/2, 1, n) 
    elif v == n:
        theta_l = stats.beta.ppf(delta/2, n, 1)       # Beta(n, 1)
        theta_u = 1.0
    else:
        theta_l = stats.beta.ppf(delta/2, v, n-v+1)
        theta_u = stats.beta.ppf(1-delta/2, v+1, n-v)
    return theta_l, theta_u

def get_ci_protection_output(alpha, epsilon_squared, eval_fn, num_evals, num_init_samples, tau=0.1, delta=0.1, tol=0.0000001, doplot=False):
    if num_evals <= 0:
        return 'inconclusive', num_evals
    num_init_samples = min(num_evals, num_init_samples)
    mses = [eval_fn(alpha)[0] for _ in range(num_init_samples)]
    num_evals -= num_init_samples
    theta_l, theta_u = get_ci_bounds(mses, epsilon_squared, delta)
    if tau > theta_u: return 'safe', num_evals
    if tau < theta_l: return 'unsafe', num_evals
    while True:
        if theta_u - theta_l < tol: return 'boundary', num_evals
        if num_evals <= 0: return 'inconclusive', num_evals
        new_mse = eval_fn(alpha)[0]
        num_evals -= 1
        mses.append(new_mse)
        theta_l, theta_u = get_ci_bounds(mses, epsilon_squared, delta)
        if tau > theta_u:
            return 'safe', num_evals
        elif tau < theta_l:
            return 'unsafe', num_evals
    
def custom_bisection_search(lo, hi , eval_fn, num_evals, epsilon_squared, mse_z_ghat_0, verbose, ci_protection_num_init_samples, 
    ci_protection_tau, ci_protection_delta, ci_protection_tol, noise_var, tol=0, log_root='ci_logs'):
    exper_str = f'noisevar={noise_var}_epssq={epsilon_squared}_numevals={num_alpha_search_iterations}_ciinit={ci_protection_num_init_samples}_cidelta={ci_protection_delta}_citau={ci_protection_tau}_citol={ci_protection_tol}'
    run_id = time.strftime("%Y%m%d_%H%M%S") + '_' + exper_str
    run_root = os.path.join(log_root, run_id)
    os.makedirs(run_root, exist_ok=True)
    is_pos = hi > 0
    lo, hi, _, num_evals = initialize_bounds(lo, hi, eval_fn, is_pos, num_evals, epsilon_squared, mse_z_ghat_0, verbose)
    attempt = 0
    alpha_list = []
    protection_status_list = []
    evals_per_alpha = []
    while lo < hi - tol and num_evals > 0:
        if verbose:
            print(f'Evals remaining: {num_evals}, lo:{lo}, hi:{hi}')
        mid = (lo + hi) / 2
        alpha_dir = os.path.join(run_root, f"attempt_{attempt:03d}_alpha_{mid:+.6f}")
        alpha_list.append(mid)
        is_protected, num_evals, evals_used_for_ci = get_ci_protection_output_logged(mid, epsilon_squared, eval_fn, num_evals, 
            ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol, log_dir=alpha_dir)
        protection_status_list.append(is_protected)
        evals_per_alpha.append(evals_used_for_ci)
        attempt += 1
        if verbose:
            print(f'Protection status for alpha={mid}: {is_protected}\n')
        if is_protected == 'unsafe':
            lo = mid
        elif is_protected == 'safe':
            hi = mid
        elif is_protected == 'boundary':
            return mid, alpha_list, protection_status_list, evals_per_alpha, run_root
        else: # is_protected == 'inconclusive'
            assert num_evals <= 0
            return (hi if is_pos else lo), alpha_list, protection_status_list, evals_per_alpha, run_root
    return (hi if is_pos else lo), alpha_list, protection_status_list, evals_per_alpha, run_root

            

# def optimize_alpha(vanilla_dy_dx, zo_dy_dx, net, criterion, method, gt_data, 
#                    label_pred, num_attack_iterations, num_dummy, imidx_list,
#                    num_alpha_search_iterations, epsilon_squared):
#      inv_attack_closure = lambda ghat: inv_attack(ghat, net, criterion, method, gt_data, label_pred, 
#                                        num_attack_iterations, None, num_dummy, None, 
#                                        imidx_list, None, False)

noise_var = 0.01
def inv_attack_closure(alpha):
    return alpha * alpha + stats.norm(loc=0, scale=np.sqrt(noise_var)).rvs(), 'z', 'ghat'
def get_bisection_search_eval_fn(sign):
    if sign == 'pos':
        def bisection_search_eval_fn(alpha):
            return inv_attack_closure(alpha)
    elif sign == 'neg':
        def bisection_search_eval_fn(alpha):
            mse, z, ghat = inv_attack_closure(alpha)
            return -mse, z, ghat
    else:
        raise ValueError('sign must be either `pos` or `neg')
    return bisection_search_eval_fn

            
epsilon_squared = 0.1
num_alpha_search_iterations = 10000
ci_protection_num_init_samples = 10
ci_protection_delta = 0.1
ci_protection_tau = 0.1
ci_protection_tol = 0.01

mse_0, z_0, ghat_0 = inv_attack_closure(0)
if mse_0 >= epsilon_squared:
    # if the vanilla gradient (alpha=0) is already larger than the error tol, then we are satisfying the constraint and can't reduce alpha any further. return 
    print( 0, ghat_0)
alpha_star_pos, search_history, status_history, eval_cnt_history, out_fold = custom_bisection_search(0, 1, get_bisection_search_eval_fn('pos'), num_alpha_search_iterations, epsilon_squared, 
                                                                               (mse_0, z_0, ghat_0), True, ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol, noise_var)
plot_sequential_alpha_progression(
    search_history,
    status_history,
    eval_cnt_history,
    epsilon_squared,
    noise_var,
    ci_protection_tau,
    out_path=os.path.join(out_fold, f'search_history_{os.path.basename(out_fold)}.png')
)
# alpha_star_neg, search_history, status_history = custom_bisection_search(-1, 0, get_bisection_search_eval_fn('neg'), num_alpha_search_iterations, -epsilon_squared, 
#                                                                                (mse_0, z_0, ghat_0), True, ci_protection_num_init_samples, ci_protection_tau, ci_protection_delta, ci_protection_tol)


Initial bounds: (0, 1)
9999/10000 evaluations remaining after initialization.
Evals remaining: 9999, lo:0, hi:1
Protection status for alpha=0.5: safe

Evals remaining: 9845, lo:0, hi:0.5
Protection status for alpha=0.25: unsafe

Evals remaining: 9835, lo:0.25, hi:0.5
Protection status for alpha=0.375: unsafe

Evals remaining: 9824, lo:0.375, hi:0.5
Protection status for alpha=0.4375: unsafe

Evals remaining: 9768, lo:0.4375, hi:0.5
Protection status for alpha=0.46875: boundary



In [50]:
os.path.basename(out_fold)

'20260219_234937'

In [ ]:
10+182+74+a+1997+3893+108+72+25+3556

9999

In [ ]:
alpha_star_pos

0.5

In [33]:
alpha_star_neg

-0.001953125